# SPU demo4

不同采样方案下，推理的性能

## 1. 加载模型

In [1]:
import jax
from helper import load_from_cache, generate, generate_topk, generate_greedy, generate_minp

base_path = "/root/.cache/huggingface/hub/models--state-spaces--mamba-130m-hf/snapshots/1e76775f628fbf1350fbe4dbb3d971ba64af25a1"
model, params, tokenizer = load_from_cache(base_path)

print("model loaded")

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.


model loaded


### 2.1: 定义运行函数

In [2]:
# 目的是加上jit
# 重要：topk非常非常慢，而且运行时间和topk的值线性相关

gen_len = 3
prompt = "Python is"
seed = 10086
topk = 40
minp = 0.1

input_ids = tokenizer.encode(prompt, return_tensors='jax')

@jax.jit
def gen_greedy(params, input_ids):
    return generate_greedy(model, params, input_ids, n_tokens_to_gen=gen_len) # 贪心采样，最快

@jax.jit
def gen_minp(params, input_ids):
    return generate_minp(model, params, input_ids, n_tokens_to_gen=gen_len, seed=seed, min_p=minp) # minp采样，比贪心慢一点点，但效果很好

@jax.jit
def gen_topk(params, input_ids):
    return generate_topk(model, params, input_ids, n_tokens_to_gen=gen_len, seed=seed, top_k=topk) # topk采样，最慢

print(f"prompt: {prompt}")
print(f"input len: {len(input_ids[0])}")
print(f"output len: {gen_len}")
print(f"seed: {seed}")
print(f"topk: {topk}")
print(f"minp: {minp}")

prompt: Python is
input len: 2
output len: 3
seed: 10086
topk: 40
minp: 0.1


### 2.2 定义模拟器

In [3]:
import sml.utils.emulation as emulation

mode = emulation.Mode.MULTIPROCESS
emulator = emulation.Emulator("3pc.json",mode)

emulator.up()

[2026-04-07 17:47:30,541]-[INFO]-[emulation.py:112]: Start multiprocess cluster...
[2026-04-07 17:47:31,141] [ForkServerProcess-4] Starting grpc server at 127.0.0.1:61923
[2026-04-07 17:47:31,143] [ForkServerProcess-1] Starting grpc server at 127.0.0.1:61920
[2026-04-07 17:47:31,169] [ForkServerProcess-3] Starting grpc server at 127.0.0.1:61922
[2026-04-07 17:47:31,171] [ForkServerProcess-5] Starting grpc server at 127.0.0.1:61924
[2026-04-07 17:47:31,177] [ForkServerProcess-2] Starting grpc server at 127.0.0.1:61921
[2026-04-07 17:47:32,628] [ForkServerProcess-1] Run : builtin_spu_init at node:0
[2026-04-07 17:47:32,629] [ForkServerProcess-2] Run : builtin_spu_init at node:1
[2026-04-07 17:47:32,629] [ForkServerProcess-3] Run : builtin_spu_init at node:2
I0407 17:47:32.637605 3108129     0 external/brpc~/src/brpc/server.cpp:1195] Server[yacl::link::transport::internal::ReceiverServiceImpl] is serving on port=61930.
W0407 17:47:32.637629 3108129     0 external/brpc~/src/brpc/server.cpp

In [4]:
@jax.jit
def warmup_fun(a, b, c):
    return a * b + c

a, b, c = 3, 2, 1
a, b, c = emulator.seal(a, b, c)
result = emulator.run(warmup_fun)(a, b, c)

result

[2026-04-07 17:47:32,803] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-07 17:47:32,805] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-07 17:47:32,806] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-07 17:47:32,810] [ForkServerProcess-4] Run : make_shares at node:3


[2026-04-07 17:47:32.902] [info] [api.cc:172] [Profiling] SPU execution warmup_fun completed, input processing took 1.23e-06s, execution took 0.000985037s, output processing took 1.98e-06s, total time 0.000988247s.
[2026-04-07 17:47:32.902] [info] [api.cc:220] HLO profiling: total time 0.00018101800000000002
[2026-04-07 17:47:32.902] [info] [api.cc:223] - pphlo.multiply, executed 1 times, duration 0.000127579s, send bytes 16 recv bytes 16, send actions 1, recv actions 1
[2026-04-07 17:47:32.902] [info] [api.cc:223] - pphlo.add, executed 1 times, duration 4.0079e-05s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-07 17:47:32.902] [info] [api.cc:223] - pphlo.free, executed 1 times, duration 1.336e-05s, send bytes 0 recv bytes 0, send actions 0, recv actions 0
[2026-04-07 17:47:32.902] [info] [api.cc:220] HAL profiling: total time 0.000117308
[2026-04-07 17:47:32.902] [info] [api.cc:223] - i_mul, executed 1 times, duration 0.000108559s, send bytes 16 recv bytes 16, se

An NVIDIA GPU may be present on this machine, but a CUDA-enabled jaxlib is not installed. Falling back to cpu.
[2026-04-07 17:47:32,868] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-07 17:47:32,870] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-07 17:47:32,872] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-07 17:47:32,874] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-07 17:47:32,875] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-07 17:47:32,892] [ForkServerProcess-1] Run : builtin_spu_run at node:0
[2026-04-07 17:47:32,892] [ForkServerProcess-2] Run : builtin_spu_run at node:1
[2026-04-07 17:47:32,892] [ForkServerProcess-3] Run : builtin_spu_run at node:2
[2026-04-07 17:47:32,895] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-07 17:47:32,895] [ForkServerProcess-4] RunR: builtin_fetch_object at node:3
[2026-04-07 17:47:32,895] [ForkServerProcess-4] RunR: builtin_fetch_object at 

array(7, dtype=int32)

### 3.1 贪心搜索


In [5]:
output_ids = gen_greedy(params, input_ids)
print(prompt, tokenizer.decode(output_ids[0]), sep='')

Python is a great tool


In [6]:
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_greedy)(s_params, s_input_ids)

[2026-04-07 17:47:40,936] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-07 17:47:40,989] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-07 17:47:40,993] [ForkServerProcess-4] Run : make_shares at node:3


[2026-04-07 17:47:40.994] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95


[2026-04-07 17:47:47,208] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-07 17:47:47,211] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-07 17:47:47,216] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-07 17:47:47,219] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-07 17:47:47,220] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-07 17:47:47,223] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-07 17:47:47,224] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-07 17:47:47,226] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-07 17:47:47,228] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-07 17:47:47,231] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-07 17:47:47,232] [ForkServerProcess-4] RunR: builtin_fetch_meta at node:3
[2026-04-07 17:47:47,234] [ForkServerProcess-4] Run : make_shares at node:3
[2026-04-07 17:47:47,243] [ForkServerProcess-4

[2026-04-07 17:48:25.730] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-07 17:48:25.731] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-07 17:48:25.732] [info] [thread_pool.cc:30] Create a fixed thread pool with size 95
[2026-04-07 17:50:38.230] [info] [api.cc:172] [Profiling] SPU execution gen_greedy completed, input processing took 7.9169e-05s, execution took 132.843331522s, output processing took 2.65e-06s, total time 132.843413341s.
[2026-04-07 17:50:38.286] [info] [api.cc:220] HLO profiling: total time 132.76322198000003
[2026-04-07 17:50:38.286] [info] [api.cc:223] - pphlo.exponential, executed 312 times, duration 42.536935856s, send bytes 4026089472 recv bytes 3741708288, send actions 22899, recv actions 21286
[2026-04-07 17:50:38.286] [info] [api.cc:223] - pphlo.convolution, executed 72 times, duration 28.625336234s, send bytes 7372800 recv bytes 7237632, send actions 406, recv actions 399
[2026-04-07 17:50:38.286] [inf

[2026-04-07 17:50:38,562] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-07 17:50:38,568] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-07 17:50:38,569] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-07 17:50:38,571] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-07 17:50:38,571] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-07 17:50:38,571] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-07 17:50:38,599] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-07 17:50:38,599] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-07 17:50:38,599] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-07 17:50:38,599] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-07 17:50:38,599] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-07 17:50:38,653] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-07 17:50:38,654] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-07 17:50

In [7]:
print(result)
print(prompt, tokenizer.decode(result[0]), sep='')

[[ 247 1270 4968]]
Python is a great tool


### 3.2 minp采样

In [8]:
output_ids = gen_minp(params, input_ids)
print(prompt, tokenizer.decode(output_ids[0]), sep='')

Python is not required.


In [9]:
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_minp)(s_params, s_input_ids)

[2026-04-07 17:50:49,062] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-07 17:50:49,175] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-07 17:50:49,180] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-07 17:50:49,181] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-07 17:50:49,181] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-07 17:50:49,182] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-07 17:50:49,182] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-07 17:50:49,187] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-07 17:50:49,187] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-07 17:50:49,187] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-07 17:50:49,189] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-07 17:50:49,189] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-07 17:50:49,193] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-07 17:50:49,193] [Fo

[2026-04-07 17:53:38.956] [info] [api.cc:172] [Profiling] SPU execution gen_minp completed, input processing took 0.000243367s, execution took 124.3432016s, output processing took 1.63e-06s, total time 124.343446597s.
[2026-04-07 17:53:39.020] [info] [api.cc:220] HLO profiling: total time 124.24386553100004
[2026-04-07 17:53:39.020] [info] [api.cc:223] - pphlo.exponential, executed 312 times, duration 41.338026825s, send bytes 4036878336 recv bytes 3765221376, send actions 22899, recv actions 21371
[2026-04-07 17:53:39.020] [info] [api.cc:223] - pphlo.convolution, executed 72 times, duration 23.793125761s, send bytes 7348224 recv bytes 7335936, send actions 404, recv actions 402
[2026-04-07 17:53:39.020] [info] [api.cc:223] - pphlo.dot_general, executed 48 times, duration 22.019825071s, send bytes 80998400 recv bytes 79824384, send actions 98422, recv actions 98438
[2026-04-07 17:53:39.020] [info] [api.cc:223] - pphlo.dot, executed 291 times, duration 15.146347967s, send bytes 26152448

[2026-04-07 17:53:39,239] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-07 17:53:39,243] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-07 17:53:39,244] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-07 17:53:39,244] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-07 17:53:39,244] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-07 17:53:39,245] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-07 17:53:39,251] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-07 17:53:39,252] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-07 17:53:39,252] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-07 17:53:39,252] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-07 17:53:39,252] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-07 17:53:39,316] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-07 17:53:39,317] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-07 17:53

In [10]:
print(result)
print(prompt, tokenizer.decode(result[0]), sep='')

[[ 247 1270 4968]]
Python is a great tool


### 3.3 topk采样

In [11]:
output_ids = gen_topk(params, input_ids)
print(prompt, tokenizer.decode(output_ids[0]), sep='')

Python is not required.


In [12]:
s_params, s_input_ids = emulator.seal(params, input_ids)
result = emulator.run(gen_topk)(s_params, s_input_ids)

[2026-04-07 17:53:49,399] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-07 17:53:49,463] [ForkServerProcess-4] Run : <lambda> at node:3
[2026-04-07 17:53:49,467] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-07 17:53:49,468] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-07 17:53:49,468] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-07 17:53:49,468] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-07 17:53:49,468] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-07 17:53:49,701] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-07 17:53:49,702] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-07 17:53:49,702] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-07 17:53:49,703] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-07 17:53:49,704] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-07 17:53:49,708] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-07 17:53:49,708] [Fo

[2026-04-07 17:56:34.908] [info] [api.cc:172] [Profiling] SPU execution gen_topk completed, input processing took 0.000253068s, execution took 122.822188422s, output processing took 4.07e-06s, total time 122.82244556s.
[2026-04-07 17:56:34.957] [info] [api.cc:220] HLO profiling: total time 122.73366145999995
[2026-04-07 17:56:34.957] [info] [api.cc:223] - pphlo.exponential, executed 312 times, duration 41.379049959s, send bytes 4026310656 recv bytes 3776016384, send actions 22899, recv actions 21307
[2026-04-07 17:56:34.957] [info] [api.cc:223] - pphlo.convolution, executed 72 times, duration 24.151585221s, send bytes 7618560 recv bytes 7753728, send actions 412, recv actions 419
[2026-04-07 17:56:34.957] [info] [api.cc:223] - pphlo.dot_general, executed 48 times, duration 22.014443714s, send bytes 83717120 recv bytes 85473536, send actions 98363, recv actions 98321
[2026-04-07 17:56:34.957] [info] [api.cc:223] - pphlo.dot, executed 291 times, duration 13.953210409s, send bytes 2592358

[2026-04-07 17:56:35,181] [ForkServerProcess-1] RunR: builtin_fetch_meta at node:0
[2026-04-07 17:56:35,185] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-07 17:56:35,186] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-07 17:56:35,187] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-07 17:56:35,187] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-07 17:56:35,187] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-07 17:56:35,209] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-07 17:56:35,210] [ForkServerProcess-3] RunR: builtin_gc at node:2
[2026-04-07 17:56:35,210] [ForkServerProcess-4] RunR: builtin_gc at node:3
[2026-04-07 17:56:35,210] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-07 17:56:35,210] [ForkServerProcess-5] RunR: builtin_gc at node:4
[2026-04-07 17:56:35,244] [ForkServerProcess-1] RunR: builtin_gc at node:0
[2026-04-07 17:56:35,245] [ForkServerProcess-2] RunR: builtin_gc at node:1
[2026-04-07 17:56

In [13]:
print(result)
print(prompt, tokenizer.decode(result[0]), sep='')

[[ 247 1270 4968]]
Python is a great tool


## 4. 停止模拟器

In [14]:
emulator.down()

[2026-04-07 17:56:35,866]-[INFO]-[emulation.py:120]: Shutdown multiprocess cluster...
